# Kratos MCP Server — materials & nonlinear plasticity walkthrough

This notebook drives the **real** `kratos-mcp` server over stdio (the same way
an AI assistant does — no mocking) to show the server's **materials** surface:
the curated **material presets** and **linear-solver presets**, the
**process-defaults introspection** parsed from Kratos source, and a small
**nonlinear** simulation — a single hexahedral element taken past its yield
point under von Mises plasticity.

It pairs with the `kratos://examples/plasticity-cube` resource and the
`docs/tutorials/plasticity-cube.md` tutorial. Every output comes from a live
run against a compiled Kratos 10.4 build with the ConstitutiveLawsApplication.

In [1]:
import asyncio
import base64
import json
import shutil
import sys
import tempfile
from pathlib import Path
from contextlib import AsyncExitStack

from IPython.display import Image as IPImage, Markdown, display

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def _preview(value, limit=4000):
    text = value if isinstance(value, str) else json.dumps(value, indent=2, default=str)
    if len(text) > limit:
        text = text[: limit] + f"\n... [truncated, {len(text)} chars total]"
    print(text)


async def call(tool_name, **kwargs):
    # Call an MCP tool, print its JSON result, display any inline images,
    # and return the parsed value for use in later cells.
    result = await session.call_tool(tool_name, kwargs)
    value = None
    for block in result.content:
        if block.type == "text":
            try:
                value = json.loads(block.text)
            except json.JSONDecodeError:
                value = block.text
        elif block.type == "image":
            display(IPImage(data=base64.b64decode(block.data), format=block.mimeType.split("/")[-1]))
    _preview(value)
    return value


async def read_resource(uri):
    result = await session.read_resource(uri)
    return result.contents[0].text


async def get_prompt(name, **kwargs):
    result = await session.get_prompt(name, kwargs)
    return result.messages[0].content.text

## Connect to the server\n\nThe server's stdio pipes are managed by anyio context managers whose cancel scopes are tied to the asyncio Task that opened them. Every Jupyter cell runs its top-level `await` in a *new* Task, so we run the whole connect → disconnect lifecycle inside one persistent background task kept alive on an `asyncio.Event`, and only *signal* it to close.

In [2]:
session = None
_stop_event = asyncio.Event()
_ready_event = asyncio.Event()


async def _server_lifecycle():
    global session
    async with AsyncExitStack() as stack:
        params = StdioServerParameters(command=sys.executable, args=["-m", "kratos_mcp.server"])
        read, write = await stack.enter_async_context(stdio_client(params))
        session = await stack.enter_async_context(ClientSession(read, write))
        await session.initialize()
        _ready_event.set()
        await _stop_event.wait()  # keeps this task (and the session) alive


_server_task = asyncio.create_task(_server_lifecycle())
await _ready_event.wait()

tools = await session.list_tools()
resources = await session.list_resources()
prompts = await session.list_prompts()
print(f"{len(tools.tools)} tools, {len(resources.resources)} resources, "
      f"{len(prompts.prompts)} prompts")
print("resources:", [str(r.uri) for r in resources.resources])

40 tools, 13 resources, 5 prompts
resources: ['kratos://docs/mdpa-format', 'kratos://docs/project-parameters', 'kratos://docs/materials', 'kratos://examples/cantilever', 'kratos://examples/thermal-bar', 'kratos://examples/naca-airfoil', 'kratos://examples/lid-driven-cavity', 'kratos://examples/plasticity-cube', 'kratos://examples/multistage-load-steps', 'kratos://examples/channel-flow', 'kratos://examples/modal-box', 'kratos://examples/dynamic-cantilever', 'kratos://examples/potential-flow']


## 1. Curated presets

The server ships **material presets** (a constitutive law + sensible default variables) and **linear-solver presets** (drop-in `linear_solver_settings` blocks), lifted from the sibling Kratos FlowGraph node library. List them:

In [3]:
await call("list_material_presets")

{
  "linear_elastic_3d": {
    "description": "Linear elastic isotropic solid (3D). Small-strain Hooke's law.",
    "application": "StructuralMechanicsApplication",
    "constitutive_law": "LinearElastic3DLaw",
    "variables": {
      "YOUNG_MODULUS": 210000000000.0,
      "POISSON_RATIO": 0.3,
      "DENSITY": 7850.0
    }
  },
  "linear_elastic_plane_strain": {
    "description": "Linear elastic isotropic solid, 2D plane-strain assumption.",
    "application": "StructuralMechanicsApplication",
    "constitutive_law": "LinearElasticPlaneStrain2DLaw",
    "variables": {
      "YOUNG_MODULUS": 210000000000.0,
      "POISSON_RATIO": 0.3,
      "DENSITY": 7850.0
    }
  },
  "linear_elastic_plane_stress": {
    "description": "Linear elastic isotropic solid, 2D plane-stress assumption (thin sheets).",
    "application": "StructuralMechanicsApplication",
    "constitutive_law": "LinearElasticPlaneStress2DLaw",
    "variables": {
      "YOUNG_MODULUS": 210000000000.0,
      "POISSON_RATIO"

{'linear_elastic_3d': {'description': "Linear elastic isotropic solid (3D). Small-strain Hooke's law.",
  'application': 'StructuralMechanicsApplication',
  'constitutive_law': 'LinearElastic3DLaw',
  'variables': {'YOUNG_MODULUS': 210000000000.0,
   'POISSON_RATIO': 0.3,
   'DENSITY': 7850.0}},
 'linear_elastic_plane_strain': {'description': 'Linear elastic isotropic solid, 2D plane-strain assumption.',
  'application': 'StructuralMechanicsApplication',
  'constitutive_law': 'LinearElasticPlaneStrain2DLaw',
  'variables': {'YOUNG_MODULUS': 210000000000.0,
   'POISSON_RATIO': 0.3,
   'DENSITY': 7850.0}},
 'linear_elastic_plane_stress': {'description': 'Linear elastic isotropic solid, 2D plane-stress assumption (thin sheets).',
  'application': 'StructuralMechanicsApplication',
  'constitutive_law': 'LinearElasticPlaneStress2DLaw',
  'variables': {'YOUNG_MODULUS': 210000000000.0,
   'POISSON_RATIO': 0.3,
   'DENSITY': 7850.0}},
 'small_strain_plasticity_von_mises_3d': {'description': 'S

In [4]:
await call("list_linear_solver_presets")

{
  "sparse_lu": {
    "description": "Direct sparse LU factorization (Eigen, LinearSolversApplication). Robust default for small/medium serial problems; no tuning.",
    "parallelism": "serial",
    "settings": {
      "solver_type": "LinearSolversApplication.sparse_lu"
    }
  },
  "skyline_lu": {
    "description": "Built-in direct skyline LU factorization (core, no LinearSolversApplication needed).",
    "parallelism": "serial",
    "settings": {
      "solver_type": "skyline_lu_factorization"
    }
  },
  "amgcl": {
    "description": "AMGCL algebraic-multigrid preconditioned Krylov solver. Scales to large serial problems; good general-purpose iterative choice.",
    "parallelism": "serial",
    "settings": {
      "solver_type": "amgcl",
      "smoother_type": "ilu0",
      "krylov_type": "gmres",
      "coarsening_type": "aggregation",
      "max_iteration": 100,
      "provide_coordinates": false,
      "gmres_krylov_space_dimension": 100,
      "verbosity": 0,
      "tolerance

{'sparse_lu': {'description': 'Direct sparse LU factorization (Eigen, LinearSolversApplication). Robust default for small/medium serial problems; no tuning.',
  'parallelism': 'serial',
  'settings': {'solver_type': 'LinearSolversApplication.sparse_lu'}},
 'skyline_lu': {'description': 'Built-in direct skyline LU factorization (core, no LinearSolversApplication needed).',
  'parallelism': 'serial',
  'settings': {'solver_type': 'skyline_lu_factorization'}},
 'amgcl': {'description': 'AMGCL algebraic-multigrid preconditioned Krylov solver. Scales to large serial problems; good general-purpose iterative choice.',
  'parallelism': 'serial',
  'settings': {'solver_type': 'amgcl',
   'smoother_type': 'ilu0',
   'krylov_type': 'gmres',
   'coarsening_type': 'aggregation',
   'max_iteration': 100,
   'provide_coordinates': False,
   'gmres_krylov_space_dimension': 100,
   'verbosity': 0,
   'tolerance': 1e-06,
   'scaling': False,
   'block_size': 1,
   'use_block_matrices_if_possible': True,

## 2. Process defaults from source

Before writing a boundary condition you need to know a process' parameters. `kratos_get_process_defaults` recovers them by parsing the process' Python source (its `ValidateAndAssignDefaults` block) — no running Kratos needed. Here is the workhorse scalar-assignment process used for Dirichlet BCs:

In [5]:
await call("kratos_get_process_defaults", python_module="assign_scalar_variable_process")

{
  "python_module": "assign_scalar_variable_process",
  "default_settings": {
    "help": "This process sets a given scalar value for a certain variable in all the nodes of a submodelpart",
    "model_part_name": "please_specify_model_part_name",
    "variable_name": "SPECIFY_VARIABLE_NAME",
    "interval": [
      0.0,
      1e+30
    ],
    "constrained": true,
    "value": {},
    "local_axes": {}
  },
  "param_types": {
    "help": "string",
    "model_part_name": "string",
    "variable_name": "string",
    "interval": "array",
    "constrained": "bool",
    "value": "json",
    "local_axes": "json"
  },
  "input_model_parts": [
    "model_part_name"
  ],
  "output_params": {
    "variable_name": "SPECIFY_VARIABLE_NAME",
    "interval": [
      0.0,
      1e+30
    ],
    "constrained": true,
    "value": {},
    "local_axes": {}
  },
  "help": "This process sets a given scalar value for a certain variable in all the nodes of a submodelpart"
}


{'python_module': 'assign_scalar_variable_process',
 'default_settings': {'help': 'This process sets a given scalar value for a certain variable in all the nodes of a submodelpart',
  'model_part_name': 'please_specify_model_part_name',
  'variable_name': 'SPECIFY_VARIABLE_NAME',
  'interval': [0.0, 1e+30],
  'constrained': True,
  'value': {},
  'local_axes': {}},
 'param_types': {'help': 'string',
  'model_part_name': 'string',
  'variable_name': 'string',
  'interval': 'array',
  'constrained': 'bool',
  'value': 'json',
  'local_axes': 'json'},
 'input_model_parts': ['model_part_name'],
 'output_params': {'variable_name': 'SPECIFY_VARIABLE_NAME',
  'interval': [0.0, 1e+30],
  'constrained': True,
  'value': {},
  'local_axes': {}},
 'help': 'This process sets a given scalar value for a certain variable in all the nodes of a submodelpart'}

## 3. The plasticity-cube example

The server ships a complete verified nonlinear example: one 1×1×1 m hexahedron stretched past yield. Read its worked-example resource (files + a verified result), then copy the real files to a working directory to run them.

In [6]:
print(await read_resource("kratos://examples/plasticity-cube"))

# Worked example: single-element von Mises plasticity (nonlinear structural)

The smallest possible demonstration of a nonlinear constitutive law. One
1x1x1 m hexahedral element (8 nodes) is stretched in x under symmetry
constraints (roller supports on the xmin/ymin/zmin faces, a prescribed x
displacement ramped on xmax). The material is small-strain isotropic von
Mises (J2) plasticity from the built-in preset
`small_strain_plasticity_von_mises_3d` (E = 210 GPa, yield stress = 250 MPa,
perfect plasticity) -- written with create_materials(preset=...). The solver
runs analysis_type "non_linear" with a residual convergence criterion.

Real files on disk (src/kratos_mcp/examples/plasticity_cube/); the mesh is the
8-node cube from mdpa_create_structured_mesh(kind='box', divisions=[1,1,1]).

## mesh.mdpa

```
Begin ModelPartData
End ModelPartData

Begin Properties 1
End Properties

Begin Nodes
    1 0 0 0
    2 1 0 0
    3 0 1 0
    4 1 1 0
    5 0 0 1
    6 1 0 1
    7 0 1 1
    8 1 1 1
End

In [7]:
import kratos_mcp  # only to locate the bundled example on disk
workdir = Path(tempfile.mkdtemp(prefix="kratos-mcp-plasticity-"))
example_dir = Path(kratos_mcp.__file__).parent / "examples" / "plasticity_cube"
for f in ("mesh.mdpa", "ProjectParameters.json", "Materials.json"):
    shutil.copy(example_dir / f, workdir / f)
print("case directory:", workdir)
await call("mdpa_inspect", path=str(workdir / "mesh.mdpa"))

case directory: /tmp/kratos-mcp-plasticity-ups4wf67
{
  "num_nodes": 8,
  "num_elements": 1,
  "num_conditions": 0,
  "elements_by_type": {
    "SmallDisplacementElement3D8N": 1
  },
  "conditions_by_type": {},
  "properties_ids": [
    1
  ],
  "bounding_box": {
    "min": [
      0.0,
      0.0,
      0.0
    ],
    "max": [
      1.0,
      1.0,
      1.0
    ]
  },
  "sub_model_parts": {
    "domain": {
      "nodes": 8,
      "elements": 1,
      "conditions": 0,
      "sub_model_parts": {}
    },
    "xmin": {
      "nodes": 4,
      "elements": 0,
      "conditions": 0,
      "sub_model_parts": {}
    },
    "xmax": {
      "nodes": 4,
      "elements": 0,
      "conditions": 0,
      "sub_model_parts": {}
    },
    "ymin": {
      "nodes": 4,
      "elements": 0,
      "conditions": 0,
      "sub_model_parts": {}
    },
    "ymax": {
      "nodes": 4,
      "elements": 0,
      "conditions": 0,
      "sub_model_parts": {}
    },
    "zmin": {
      "nodes": 4,
      "elements"

{'num_nodes': 8,
 'num_elements': 1,
 'num_conditions': 0,
 'elements_by_type': {'SmallDisplacementElement3D8N': 1},
 'conditions_by_type': {},
 'properties_ids': [1],
 'bounding_box': {'min': [0.0, 0.0, 0.0], 'max': [1.0, 1.0, 1.0]},
 'sub_model_parts': {'domain': {'nodes': 8,
   'elements': 1,
   'conditions': 0,
   'sub_model_parts': {}},
  'xmin': {'nodes': 4, 'elements': 0, 'conditions': 0, 'sub_model_parts': {}},
  'xmax': {'nodes': 4, 'elements': 0, 'conditions': 0, 'sub_model_parts': {}},
  'ymin': {'nodes': 4, 'elements': 0, 'conditions': 0, 'sub_model_parts': {}},
  'ymax': {'nodes': 4, 'elements': 0, 'conditions': 0, 'sub_model_parts': {}},
  'zmin': {'nodes': 4, 'elements': 0, 'conditions': 0, 'sub_model_parts': {}},
  'zmax': {'nodes': 4, 'elements': 0, 'conditions': 0, 'sub_model_parts': {}}}}

## 4. Build the material from a preset

The example's `Materials.json` was written with `create_materials(preset=...)`. Reproduce it: the preset fills the von Mises plasticity law and its default variables (E, yield stress, fracture energy, hardening) in one call — override just what you need.

In [8]:
await call(
    "create_materials",
    output_file=str(workdir / "Materials.json"),
    materials=[{
        "model_part_name": "Structure.domain",
        "preset": "small_strain_plasticity_von_mises_3d",
    }],
)

{
  "written_to": "/tmp/kratos-mcp-plasticity-ups4wf67/Materials.json",
  "materials": {
    "properties": [
      {
        "model_part_name": "Structure.domain",
        "properties_id": 1,
        "Material": {
          "Variables": {
            "YOUNG_MODULUS": 210000000000.0,
            "POISSON_RATIO": 0.3,
            "DENSITY": 7850.0,
            "YIELD_STRESS": 250000000.0,
            "FRACTURE_ENERGY": 1000000.0,
            "HARDENING_CURVE": 3
          },
          "Tables": {},
          "constitutive_law": {
            "name": "SmallStrainIsotropicPlasticity3DVonMisesVonMises"
          }
        }
      }
    ]
  }
}


{'written_to': '/tmp/kratos-mcp-plasticity-ups4wf67/Materials.json',
 'materials': {'properties': [{'model_part_name': 'Structure.domain',
    'properties_id': 1,
    'Material': {'Variables': {'YOUNG_MODULUS': 210000000000.0,
      'POISSON_RATIO': 0.3,
      'DENSITY': 7850.0,
      'YIELD_STRESS': 250000000.0,
      'FRACTURE_ENERGY': 1000000.0,
      'HARDENING_CURVE': 3},
     'Tables': {},
     'constitutive_law': {'name': 'SmallStrainIsotropicPlasticity3DVonMisesVonMises'}}}]}}

## 5. Validate and run the nonlinear case

The solver runs `analysis_type: "non_linear"` with a residual convergence criterion — each load step is a Newton–Raphson solve.

In [9]:
await call("validate_case", case_dir=str(workdir))

{
  "valid": false,
  "issues": [
    "solver_settings validation failed: Error: The item with name \"line_search\" is present in this Parameters but NOT in the default values\nHence Validation fails\nParameters being validated are : \n{\n    \"analysis_type\": \"non_linear\",\n    \"convergence_criterion\": \"residual_criterion\",\n    \"displacement_absolute_tolerance\": 1e-09,\n    \"displacement_relative_tolerance\": 0.0001,\n    \"domain_size\": 3,\n    \"echo_level\": 0,\n    \"line_search\": false,\n    \"linear_solver_settings\": {\n        \"solver_type\": \"LinearSolversApplication.sparse_lu\"\n    },\n    \"material_import_settings\": {\n        \"materials_filename\": \"Materials.json\"\n    },\n    \"max_iteration\": 20,\n    \"model_import_settings\": {\n        \"input_filename\": \"mesh\",\n        \"input_type\": \"mdpa\"\n    },\n    \"model_part_name\": \"Structure\",\n    \"residual_absolute_tolerance\": 1e-09,\n    \"residual_relative_tolerance\": 0.0001,\n    \"ro

{'valid': False,
 'issues': ['solver_settings validation failed: Error: The item with name "line_search" is present in this Parameters but NOT in the default values\nHence Validation fails\nParameters being validated are : \n{\n    "analysis_type": "non_linear",\n    "convergence_criterion": "residual_criterion",\n    "displacement_absolute_tolerance": 1e-09,\n    "displacement_relative_tolerance": 0.0001,\n    "domain_size": 3,\n    "echo_level": 0,\n    "line_search": false,\n    "linear_solver_settings": {\n        "solver_type": "LinearSolversApplication.sparse_lu"\n    },\n    "material_import_settings": {\n        "materials_filename": "Materials.json"\n    },\n    "max_iteration": 20,\n    "model_import_settings": {\n        "input_filename": "mesh",\n        "input_type": "mdpa"\n    },\n    "model_part_name": "Structure",\n    "residual_absolute_tolerance": 1e-09,\n    "residual_relative_tolerance": 0.0001,\n    "rotation_dofs": false,\n    "solver_type": "Static",\n    "time_

In [10]:
run = await call("run_simulation", case_dir=str(workdir), wait_seconds=60)
job_id = run["job_id"]

{
  "job_id": "20260714-172346-418404",
  "case_dir": "/tmp/kratos-mcp-plasticity-ups4wf67",
  "parameters_file": "ProjectParameters.json",
  "state": "succeeded",
  "pid": 2850175,
  "returncode": 0,
  "created_at": 1784042626.652905,
  "started_at": 1784042626.6529052,
  "finished_at": 1784042628.6557107,
  "analysis_type": null,
  "extra": {},
  "elapsed_seconds": 2.0
}


In [11]:
await call("job_status", job_id=job_id)

{
  "job_id": "20260714-172346-418404",
  "case_dir": "/tmp/kratos-mcp-plasticity-ups4wf67",
  "parameters_file": "ProjectParameters.json",
  "state": "succeeded",
  "pid": 2850175,
  "returncode": 0,
  "created_at": 1784042626.652905,
  "started_at": 1784042626.6529052,
  "finished_at": 1784042628.6557107,
  "analysis_type": null,
  "extra": {},
  "elapsed_seconds": 2.0,
  "progress": {
    "current_step": 5,
    "current_time": 5.0,
    "num_steps_seen": 5,
    "errors_detected": []
  }
}


{'job_id': '20260714-172346-418404',
 'case_dir': '/tmp/kratos-mcp-plasticity-ups4wf67',
 'parameters_file': 'ProjectParameters.json',
 'state': 'succeeded',
 'pid': 2850175,
 'returncode': 0,
 'created_at': 1784042626.652905,
 'started_at': 1784042626.6529052,
 'finished_at': 1784042628.6557107,
 'analysis_type': None,
 'extra': {},
 'elapsed_seconds': 2.0,
 'progress': {'current_step': 5,
  'current_time': 5.0,
  'num_steps_seen': 5,
  'errors_detected': []}}

## 6. The elastic → plastic transition

The x-displacement on the `xmax` face is ramped 0.001 m per step. Read the reaction force in x on the fixed `xmin` face at each step (Newton's third law: the support reaction balances the internal stress). Below the yield strain the response is linear elastic; past it, the reaction plateaus at σ_y·A — the material yields instead of carrying more load.

In [12]:
import glob, meshio, numpy as np
vtks = sorted(glob.glob(str(workdir / "vtk_output" / "*.vtk")),
              key=lambda p: int("".join(filter(str.isdigit, Path(p).stem)) or 0))
print("step  imposed_ux   |Fx| (reaction)   regime")
for f in vtks:
    m = meshio.read(f); pts = np.asarray(m.points)
    R = np.asarray(m.point_data["REACTION"]); D = np.asarray(m.point_data["DISPLACEMENT"])
    ux = float(D[pts[:, 0] > 0.999, 0].mean())
    Fx = abs(float(R[pts[:, 0] < 1e-6, 0].sum()))
    regime = "elastic" if Fx < 2.49e8 else "YIELDED (plateau at sigma_y*A)"
    print(f"      {ux:6.4f}     {Fx:.3e} N   {regime}")

step  imposed_ux   |Fx| (reaction)   regime
      0.0010     2.100e+08 N   elastic
      0.0020     2.500e+08 N   YIELDED (plateau at sigma_y*A)
      0.0030     2.500e+08 N   YIELDED (plateau at sigma_y*A)
      0.0040     2.500e+08 N   YIELDED (plateau at sigma_y*A)
      0.0050     2.500e+08 N   YIELDED (plateau at sigma_y*A)


The reaction climbs elastically to **2.1e8 N** at strain 0.001 (= E·ε·A = 210 MPa), then locks at **2.5e8 N** = σ_y·A = 250 MPa·1 m² once yielded — textbook perfect plasticity. A `LinearElastic3DLaw` would have kept climbing; the plateau *is* the nonlinearity.

## Cleanup\n\nClose the client session (which stops the server subprocess). The case directory is left on disk — open its `vtk_output/` in ParaView to inspect the raw results yourself.

In [13]:
_stop_event.set()      # wake the lifecycle task up ...
await _server_task     # ... and let it exit its own AsyncExitStack from the task that opened it
print("session closed, server subprocess terminated")
print("case directory left on disk:", workdir)

session closed, server subprocess terminated
case directory left on disk: /tmp/kratos-mcp-plasticity-ups4wf67
